## Context

LangChain aims to make it easy to build LLM applications. One type of LLM application we can build is an agent. There’s a lot of excitement around building agents because they can **automate a wide range of tasks** that were previously impossible.

In practice though, it is incredibly **difficult** to build systems that **reliably** execute on these tasks. LangChain has worked with their users to put agents into production, they’ve learned that **more control** is often necessary. We might need an agent to always call a specific tool first or use different prompts based on its state.

To tackle this problem, they’ve built **LangGraph** — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to **help developers add better precision and control into agent workflows**, suitable for the complexity of real-world systems.

## Chat models 

We'll use Chat Models, which take a sequence of messages as input and return messages as output. 

LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use [ChatGoogleGenerativeAI](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai) because it is both popular and performant.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_2dot5_chat = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0,
    max_tokens = None,
    timeout = None,
    max_retries = 2,
    # other params...
)

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain-core/runnables). For the most part, we'll be using:

- [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
- [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

Note: Chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a **role** (that describes who is saying the message) and a **content** property.

In [3]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Dibyajyoti")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
gemini_2dot5_chat.invoke(messages)

AIMessage(content='Hello there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cff98-9371-74d0-9bb1-3e9fc9cc4cc3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 10, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}})

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.

In [4]:
gemini_2dot5_chat.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cff98-ac6b-7a41-b65c-501c200371c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

## Search Tools

[Tavily](https://app.tavily.com/home) is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results.

In [5]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})

search_docs = data.get("results", data)
search_docs

[{'url': 'https://www.ibm.com/think/topics/langgraph',
  'title': 'What is LangGraph? - IBM',
  'content': 'LangGraph, created by [LangChain](https://www.ibm.com/think/topics/langchain), is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an [AI agent workflow](https://www.ibm.com/think/topics/ai-agents). The following example can offer a clearer understanding of LangGraph: Think about these graph-based architectures as a powerful configurable map, a “Super-Map.” Users can envision the [AI workflow](https://www.ibm.com/think/topics/ai-workflow) as being “The Navigator” of this “Super-Map.” Finally, in this example, the user is “The Cartographer.” In this sense, the navigator charts out the optimal routes between points on the “Super-Map,” all of which are created by “The Cartographer